# 7.26 正解移除後，示範答案應該怎麼變？


材料寫「要查卡片進度，申請編號K17」，候選有「查卡片進度」「查地址變更規則」「詢問申請編號」「以上皆不適合」。助手示範應回答「查卡片進度」。現在只刪掉第一個候選，材料和問題都沒變，還能沿用原答案嗎？不能：這次要求從**目前列表**作答，示範也必須跟著列表改。

前置是[7.3–7.5的回答位置與代價遮罩](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)、[7.11的監督式微調](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)與[7.24的條件和示範配對](https://birdhackor.github.io/tiny-perceptron-vlm/7.24.html)。這裡繼續用文字回答，先把應教的行為寫對。

先訂一個可核對的教學世界：查卡片進度需要申請編號；查地址變更規則不需要。編號沒給，就先詢問；重設密碼不在支援範圍。每個候選有固定ID，方便資料程式追蹤同一動作；ID不是列表位置，也不要求助手輸出ID。候選描述必須自己說清意思，不能寫「同上」。

「以上皆不適合」（none）表示**目前實質候選裡沒有符合問題的答案**。它可能因為正解被刪掉，也可能因為任務不在支援範圍；只看到none，不能說模型已分清原因。資料可另存原因供稽核，不能把原因偷放入模型輸入。

| 只改哪裡 | 應給助手的文字示範 |
| --- | --- |
| 完整材料、完整列表 | 查卡片進度 |
| 只刪查卡片進度 | 以上皆不適合 |
| 保留正解、增加無關候選 | 查卡片進度 |
| 刪掉材料中的申請編號，保留詢問候選 | 詢問申請編號 |

下面是完整的離線資料建構程式。`needed`先依事實求出動作；`make`再檢查該動作是否在場。它產生文字示範，沒有呼叫或訓練模型。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
meaning = {
    "card": "查卡片進度",
    "address": "查地址變更規則",
    "ask_id": "詢問申請編號",
    "weather": "查天氣",
    "none": "以上皆不適合",
}
rules = "查卡片進度要申請編號；未提供就先詢問。查地址變更規則不需編號。其餘任務不支援。"
question = "依規則，下一步選哪個候選？只輸出描述；沒有適合動作就輸出以上皆不適合。"


def make(state, order):
    if state["任務"] == "查卡片進度":
        needed = "card" if "申請編號" in state else "ask_id"
    elif state["任務"] == "查地址變更規則":
        needed = "address"
    else:
        needed = None  # 不支援；不替它編造可執行的動作
    chosen = needed if needed in order else "none"
    reason = "不支援" if needed is None else ("列表缺正解" if chosen == "none" else "有合適候選")
    facts = "；".join(f"{k}={v}" for k, v in state.items())
    options = "／".join(meaning[k] for k in order)
    prompt = f"規則：{rules}\n材料：{facts}\n問題：{question}\n候選：{options}"
    return {"prompt": prompt, "answer": meaning[chosen], "audit_reason": reason}


state = {"任務": "查卡片進度", "申請編號": "K17"}
full = ["card", "address", "ask_id", "none"]
rows = [
    ("原題", make(state, full)),
    ("移除正解", make(state, ["address", "ask_id", "none"])),
    ("增加干擾", make(state, full + ["weather"])),
    ("重排", make(state, list(reversed(full)))),
    ("缺編號", make({"任務": "查卡片進度"}, full)),
    ("不支援", make({"任務": "重設密碼"}, full)),
]
for name, row in rows:
    print(name, "→", row["answer"], "／稽核：", row["audit_reason"])
print("移除正解的實際輸入：\n" + rows[1][1]["prompt"])

`facts`用`"；".join(...)`把每個「欄名=值」以分號接成材料字串；`options`同樣以斜線接起候選描述。`list(reversed(full))`把原列表倒過來，所以「重排」那筆只換候選出現順序，內容沒有增刪。

前六行會是：

```text
原題 → 查卡片進度 ／稽核： 有合適候選
移除正解 → 以上皆不適合 ／稽核： 列表缺正解
增加干擾 → 查卡片進度 ／稽核： 有合適候選
重排 → 查卡片進度 ／稽核： 有合適候選
缺編號 → 詢問申請編號 ／稽核： 有合適候選
不支援 → 以上皆不適合 ／稽核： 不支援
```

原題與移除正解這兩筆叫**最小配對**：只改一項有因果關係的條件，讓答案變化有清楚理由。兩筆都有none，卻只有移除正解那筆要選它，可避免把「看到none就回答none」寫成資料捷徑。缺編號那筆則改了事實，教的是先詢問，不能和刪候選混為同一種變化。

送進SFT時，`prompt`放使用者內容，`answer`放助手內容；沿用7.3的做法，問題位置不計代價，助手回答與結束標記才是目標。`audit_reason`只供檢查，不送入對話。也要刪除語義重複的候選，保持每題唯一正解。同一情境的刪候選、重排、缺欄與換措辭版本都放同一個資料家族，不能分到訓練與測試兩邊，冒充新情境。

練習：把「缺編號」那筆的`full`換成`["card", "address", "none"]`。先寫出示範及稽核原因，再執行。應得到「以上皆不適合／列表缺正解」：缺資訊時需要的詢問動作也被拿掉了。增加這些資料是否真的讓模型更會判斷，仍需訓練後用留出的情境檢查。

來源：候選重排、增干擾與正解移除配對可對照[Kev固定版本的`augment`與`none_pair`](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/data.py#L313-L360)。本節規則與文字SFT示範為自製，沒有使用Kev權重、資料或模型輸出。
